In [0]:
# Step 1 - Check all our uploaded tables exist
tables = spark.sql("SHOW TABLES")
tables.show()

+--------+-------------+-----------+
|database|    tableName|isTemporary|
+--------+-------------+-----------+
| default| customerscsv|      false|
| default|customersjson|      false|
| default|    orderscsv|      false|
| default|   ordersjson|      false|
| default|  productscsv|      false|
| default| productsjson|      false|
+--------+-------------+-----------+



In [0]:
# Step 2 - BRONZE LAYER
# Read all 6 tables and just display them to check the data

# Customers
df_customers_csv = spark.read.table("customerscsv")
df_customers_json = spark.read.table("customersjson")

# Orders
df_orders_csv = spark.read.table("orderscsv")
df_orders_json = spark.read.table("ordersjson")

# Products
df_products_csv = spark.read.table("productscsv")
df_products_json = spark.read.table("productsjson")

print("Customers CSV count:", df_customers_csv.count())
print("Customers JSON count:", df_customers_json.count())
print("Orders CSV count:", df_orders_csv.count())
print("Orders JSON count:", df_orders_json.count())
print("Products CSV count:", df_products_csv.count())
print("Products JSON count:", df_products_json.count())

Customers CSV count: 5
Customers JSON count: 5
Orders CSV count: 5
Orders JSON count: 5
Products CSV count: 4
Products JSON count: 4


In [0]:
# Step 3 - BRONZE LAYER
# Combine CSV and JSON together for each dataset
# and save as Bronze Delta tables

from pyspark.sql.functions import current_timestamp

# --- CUSTOMERS BRONZE ---
df_bronze_customers = df_customers_csv.unionByName(df_customers_json)
df_bronze_customers = df_bronze_customers.withColumn("ingested_at", current_timestamp())
df_bronze_customers.write.format("delta").mode("overwrite").saveAsTable("bronze_customers")
print("Bronze Customers saved! Total rows:", df_bronze_customers.count())

# --- ORDERS BRONZE ---
df_bronze_orders = df_orders_csv.unionByName(df_orders_json)
df_bronze_orders = df_bronze_orders.withColumn("ingested_at", current_timestamp())
df_bronze_orders.write.format("delta").mode("overwrite").saveAsTable("bronze_orders")
print("Bronze Orders saved! Total rows:", df_bronze_orders.count())

# --- PRODUCTS BRONZE ---
df_bronze_products = df_products_csv.unionByName(df_products_json)
df_bronze_products = df_bronze_products.withColumn("ingested_at", current_timestamp())
df_bronze_products.write.format("delta").mode("overwrite").saveAsTable("bronze_products")
print("Bronze Products saved! Total rows:", df_bronze_products.count())

Bronze Customers saved! Total rows: 10
Bronze Orders saved! Total rows: 10
Bronze Products saved! Total rows: 8


In [0]:
# Step 4 - SILVER LAYER
# Clean the data - remove duplicates, fix column names, drop nulls

from pyspark.sql.functions import col, trim, upper, lower

# --- CUSTOMERS SILVER ---
df_silver_customers = df_bronze_customers \
    .dropDuplicates(["customer_id"]) \
    .dropna(subset=["customer_id", "first_name"]) \
    .withColumn("first_name", trim(upper(col("first_name")))) \
    .withColumn("last_name", trim(upper(col("last_name"))))

df_silver_customers.write.format("delta").mode("overwrite").saveAsTable("silver_customers")
print("Silver Customers saved! Total rows:", df_silver_customers.count())

# --- ORDERS SILVER ---
df_silver_orders = df_bronze_orders \
    .dropDuplicates(["order_id"]) \
    .dropna(subset=["order_id", "customer_id"])

df_silver_orders.write.format("delta").mode("overwrite").saveAsTable("silver_orders")
print("Silver Orders saved! Total rows:", df_silver_orders.count())

# --- PRODUCTS SILVER ---
df_silver_products = df_bronze_products \
    .dropDuplicates(["product_id"]) \
    .dropna(subset=["product_id", "product_name"]) \
    .withColumn("product_name", trim(upper(col("product_name"))))

df_silver_products.write.format("delta").mode("overwrite").saveAsTable("silver_products")
print("Silver Products saved! Total rows:", df_silver_products.count())

Silver Customers saved! Total rows: 5
Silver Orders saved! Total rows: 5
Silver Products saved! Total rows: 4


In [0]:
# Check exact column names in each silver table
print("Orders columns:", df_silver_orders.columns)
print("Customers columns:", df_silver_customers.columns)
print("Products columns:", df_silver_products.columns)

Orders columns: ['order_id', 'customer_id', 'product_id', 'order_date', 'quantity', 'total_amount', 'ingested_at']
Customers columns: ['customer_id', 'first_name', 'last_name', 'country', 'signup_date', 'ingested_at']
Products columns: ['product_id', 'product_name', 'category', 'price', 'ingested_at']


In [0]:
# Step 5 - GOLD LAYER (Fixed)
# Join customers + orders + products together
# and create a final business summary table

from pyspark.sql.functions import sum, count, round

# Join orders with customers
df_orders_customers = df_silver_orders.join(
    df_silver_customers,
    on="customer_id",
    how="left"
)

# Join with products
df_full = df_orders_customers.join(
    df_silver_products,
    on="product_id",
    how="left"
)

# Create Gold summary - total spending per customer
df_gold = df_full.groupBy(
    "customer_id",
    "first_name",
    "last_name"
) \
.agg(
    count("order_id").alias("total_orders"),
    round(sum("total_amount"), 2).alias("total_spent")
) \
.orderBy("total_spent", ascending=False)

df_gold.write.format("delta").mode("overwrite").saveAsTable("gold_customer_summary")
print("Gold Layer saved!")
df_gold.show()

Gold Layer saved!
+-----------+----------+---------+------------+-----------+
|customer_id|first_name|last_name|total_orders|total_spent|
+-----------+----------+---------+------------+-----------+
|        101|     ALICE|   MORGAN|           2|        330|
|        105|      EMMA|    PATEL|           1|        180|
|        103|     CARLA|     ZHAO|           1|         90|
|        102|       BEN|    SINGH|           1|         45|
+-----------+----------+---------+------------+-----------+



In [0]:
# Query your Gold table using SQL
spark.sql("""
    SELECT 
        first_name,
        last_name,
        total_orders,
        total_spent,
        RANK() OVER (ORDER BY total_spent DESC) as customer_rank
    FROM gold_customer_summary
""").show()

+----------+---------+------------+-----------+-------------+
|first_name|last_name|total_orders|total_spent|customer_rank|
+----------+---------+------------+-----------+-------------+
|     ALICE|   MORGAN|           2|        330|            1|
|      EMMA|    PATEL|           1|        180|            2|
|     CARLA|     ZHAO|           1|         90|            3|
|       BEN|    SINGH|           1|         45|            4|
+----------+---------+------------+-----------+-------------+



In [0]:
# Step 6 - DATA QUALITY CHECKS
# Check for nulls and duplicates in each silver table

print("=== DATA QUALITY REPORT ===")
print()

# Customers quality check
cust_total = df_silver_customers.count()
cust_nulls = df_silver_customers.filter(
    col("customer_id").isNull() | col("first_name").isNull()
).count()
cust_dupes = cust_total - df_silver_customers.dropDuplicates(["customer_id"]).count()
print(f"Customers → Total: {cust_total} | Nulls: {cust_nulls} | Duplicates: {cust_dupes}")

# Orders quality check
ord_total = df_silver_orders.count()
ord_nulls = df_silver_orders.filter(
    col("order_id").isNull() | col("customer_id").isNull()
).count()
ord_dupes = ord_total - df_silver_orders.dropDuplicates(["order_id"]).count()
print(f"Orders    → Total: {ord_total} | Nulls: {ord_nulls} | Duplicates: {ord_dupes}")

# Products quality check
prod_total = df_silver_products.count()
prod_nulls = df_silver_products.filter(
    col("product_id").isNull() | col("product_name").isNull()
).count()
prod_dupes = prod_total - df_silver_products.dropDuplicates(["product_id"]).count()
print(f"Products  → Total: {prod_total} | Nulls: {prod_nulls} | Duplicates: {prod_dupes}")

print()
print("=== ALL CHECKS PASSED ✅ ===")

=== DATA QUALITY REPORT ===

Customers → Total: 5 | Nulls: 0 | Duplicates: 0
Orders    → Total: 5 | Nulls: 0 | Duplicates: 0
Products  → Total: 4 | Nulls: 0 | Duplicates: 0

=== ALL CHECKS PASSED ✅ ===


In [0]:
# Step 7 - VISUALIZE Gold Layer
# Show the final summary as a nice table with chart

df_final = spark.sql("""
    SELECT 
        first_name || ' ' || last_name as customer_name,
        total_orders,
        total_spent,
        RANK() OVER (ORDER BY total_spent DESC) as customer_rank
    FROM gold_customer_summary
    ORDER BY total_spent DESC
""")

df_final.show()

+-------------+------------+-----------+-------------+
|customer_name|total_orders|total_spent|customer_rank|
+-------------+------------+-----------+-------------+
| ALICE MORGAN|           2|        330|            1|
|   EMMA PATEL|           1|        180|            2|
|   CARLA ZHAO|           1|         90|            3|
|    BEN SINGH|           1|         45|            4|
+-------------+------------+-----------+-------------+

